### **Memuat Dataset**

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("DataFiles/data_cleaning.csv")

import pandas as pd

# --- 1. Membuat DataFrame untuk Ringkasan Dimensi ---
summary_data = {
    "Metrik": ["Total Baris", "Total Kolom"],
    "Nilai": [df.shape[0], df.shape[1]]
}
df_summary = pd.DataFrame(summary_data)

# --- 2. Membuat DataFrame untuk Audit Struktur Kolom ---
dup_cols = df.columns[df.columns.duplicated()].tolist()
whitespace_cols = [c for c in df.columns if c != c.strip()]

audit_data = {
    "Jenis Pengecekan": ["Status Duplikasi", "Status Spasi Tersembunyi"],
    "Status": [
        "✅ Aman" if not dup_cols else "⚠️ Terdeteksi",
        "✅ Aman" if not whitespace_cols else "⚠️ Terdeteksi"
    ],
    "Detail Kolom": [
        ", ".join(dup_cols) if dup_cols else "-",
        ", ".join(whitespace_cols) if whitespace_cols else "-"
    ]
}
df_audit = pd.DataFrame(audit_data)

# --- 3. Menampilkan Semua Output ---
print("📊 RINGKASAN DATASET")
display(df_summary)

print("\n🔍 AUDIT STRUKTUR KOLOM")
display(df_audit)

print("\n👀 SAMPEL 2 DATA TERATAS")
display(df.head(2))

📊 RINGKASAN DATASET


,Metrik,Nilai
0,Total Baris,11429
1,Total Kolom,83



🔍 AUDIT STRUKTUR KOLOM


,Jenis Pengecekan,Status,Detail Kolom
0,Status Duplikasi,✅ Aman,-
1,Status Spasi Tersembunyi,✅ Aman,-



👀 SAMPEL 2 DATA TERATAS


,url,length_url,length_hostname,ip,nb_dots,nb_hyphens,nb_at,nb_qm,nb_and,nb_eq,...,domain_in_title,domain_with_copyright,whois_registered_domain,domain_registration_length,domain_age,web_traffic,dns_record,google_index,page_rank,label
0,http://www.crestonwood.com/router.php,37,19,0,3,0,0,0,0,0,...,0,1,0,45,-1,0,1,1,4,0
1,http://shadetreetechnology.com/V4/validation/a...,77,23,1,1,0,0,0,0,0,...,1,0,0,77,5767,0,0,1,2,1


## **3. Split Data**

In [2]:
from sklearn.model_selection import train_test_split

# 1. PERSIAPAN KOLOM & MODE FITUR
df.columns = df.columns.str.strip()

TARGET_COL = "label"
ACTIVE_TRAIN_FEATURES = "hybrid81"  # "url37" atau "hybrid81"

url_features_37 = [
    "length_url", "length_hostname", "ip", "nb_dots", "nb_hyphens", "nb_at", "nb_qm", "nb_and",
    "nb_eq", "nb_underscore", "nb_tilde", "nb_percent", "nb_slash", "nb_star", "nb_colon",
    "nb_comma", "nb_semicolumn", "nb_dollar", "nb_space", "nb_www", "nb_com", "nb_dslash",
    "http_in_path", "https_token", "ratio_digits_url", "ratio_digits_host", "punycode", "port",
    "tld_in_path", "tld_in_subdomain", "abnormal_subdomain", "nb_subdomains", "prefix_suffix",
    "random_domain", "shortening_service", "path_extension", "nb_redirection"
]

id_cols = [c for c in ["url"] if c in df.columns]
all_features = [c for c in df.columns if c not in [TARGET_COL] + id_cols]

webcontent_features_44 = [c for c in all_features if c not in url_features_37]
hybrid_features_81 = url_features_37 + webcontent_features_44

feature_candidates = hybrid_features_81 if ACTIVE_TRAIN_FEATURES == "hybrid81" else url_features_37

# 2. SIAPKAN X DAN y (NUMERIK)
X_all = df[feature_candidates].apply(pd.to_numeric, errors="coerce")
y_all = pd.to_numeric(df[TARGET_COL], errors="coerce").fillna(0).astype("int64")

# 3. SPLIT 80:20 (STRATIFY, RANDOM_STATE=12)
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.2,
    stratify=y_all,
    random_state=12
)

# 4. SIMPAN FITUR VALID (NON-NaN), LALU FILTER
selected_features = X_train.columns[X_train.notna().any()].tolist()
X_train = X_train[selected_features]
X_test = X_test[selected_features]

# 5. IMPUTASI MISSING VALUE PAKAI MEDIAN DARI TRAIN
med_all = X_train.median(numeric_only=True)
X_train = X_train.fillna(med_all)
X_test = X_test.fillna(med_all)

# 6. OUTPUT
df_config = pd.DataFrame({
    "Pengaturan Pemodelan": [
        "Kolom Target", "Mode Pelatihan",
        "Total Kandidat Fitur", "Jumlah Fitur Valid",
        "Strategi Imputasi"
    ],
    "Keterangan": [
        TARGET_COL, ACTIVE_TRAIN_FEATURES,
        len(feature_candidates), len(selected_features),
        "Median (dari Data Train)"
    ]
})

df_split = pd.DataFrame({
    "Dataset": ["Train Data (80%)", "Test Data (20%)"],
    "Jumlah Baris": [X_train.shape[0], X_test.shape[0]],
    "Jumlah Kolom (Fitur)": [X_train.shape[1], X_test.shape[1]]
})

print("⚙️ KONFIGURASI:")
display(df_config)

print("\n📊 DIMENSI DATASET:")
display(df_split)

⚙️ KONFIGURASI:


,Pengaturan Pemodelan,Keterangan
0,Kolom Target,label
1,Mode Pelatihan,hybrid81
2,Total Kandidat Fitur,81
3,Jumlah Fitur Valid,81
4,Strategi Imputasi,Median (dari Data Train)



📊 DIMENSI DATASET:


,Dataset,Jumlah Baris,Jumlah Kolom (Fitur)
0,Train Data (80%),9143,81
1,Test Data (20%),2286,81


### **Rule-Based Filtering (Pre-filter)**

In [3]:
import re
import math
import ipaddress
from urllib.parse import urlparse, urljoin
import tldextract
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

SHORTENERS = {
    "bit.ly", "goo.gl", "tinyurl.com", "ow.ly", "t.co", "is.gd", "buff.ly",
    "adf.ly", "bit.do", "cutt.ly"
}

SUSPICIOUS_TLD = [
    "zip", "xyz", "top", "tk", "ga", "ml", "gq", "cf", "pw", "cc", "club",
    "ws", "biz", "online", "site", "live", "work", "icu", "info",
    "cn", "ru", "loan", "download", "click"
]

STANDARD_PORTS = {21, 22, 23, 80, 443, 445, 1433, 1521, 3306, 3389}
PHISH_HINTS = [
    "login", "verify", "update", "secure", "account", "bank",
    "paypal", "apple", "microsoft", "confirm", "signin", "password"
]
BRANDS = [
    "google", "facebook", "apple", "microsoft", "amazon", "paypal",
    "instagram", "whatsapp", "telegram", "netflix", "github", "linkedin"
]
PREFILTER_HARD_PHISHING_SCORE = 7

try:
    data = df.copy()
except NameError:
    print("⚠️ Peringatan: Variabel 'df' belum didefinisikan. Jalankan cell 'Memuat Dataset' dulu.")
    data = None

def entropy(s):
    if not s:
        return 0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs)

def parse_url(url):
    u = (url or "").strip()
    if not u.startswith(("http://", "https://")):
        u = "http://" + u
    parsed = urlparse(u)
    hostname = (parsed.hostname or "").lower()
    path = parsed.path or ""
    return u, parsed, hostname, path

def is_ip(hostname):
    try:
        ipaddress.ip_address(hostname)
        return 1
    except Exception:
        return 0

def extract_url_features(url):
    full, parsed, hostname, path = parse_url(url)
    ext = tldextract.extract(full)
    subdomain = (ext.subdomain or "").lower()
    domain = (ext.domain or "").lower()
    suffix = (ext.suffix or "").lower()

    digits_url = sum(c.isdigit() for c in full)
    digits_host = sum(c.isdigit() for c in hostname)

    random_domain = 1 if entropy(domain) > 3.5 else 0
    shortening_service = 1 if any(s in hostname for s in SHORTENERS) else 0
    prefix_suffix = 1 if "-" in domain else 0
    path_extension = 1 if "." in path.split("/")[-1] else 0
    nb_redirection = max(full.lower().count("http") - 1, 0)

    try:
        parsed_port = parsed.port
    except ValueError:
        parsed_port = None
    port_flag = 1 if (parsed_port is not None and parsed_port not in STANDARD_PORTS) else 0

    tld_last_label = suffix.split(".")[-1] if suffix else ""
    suspicious_tld_flag = 1 if (suffix in SUSPICIOUS_TLD or tld_last_label in SUSPICIOUS_TLD) else 0

    domain_in_brand = 1 if any(b in domain for b in BRANDS) else 0
    brand_in_subdomain = 1 if any(b in subdomain for b in BRANDS) else 0
    brand_in_path = 1 if any(b in path.lower() for b in BRANDS) else 0

    statistical_report = 1 if (
        suspicious_tld_flag == 1
        or is_ip(hostname) == 1
        or full.count("@") >= 1
        or random_domain == 1
    ) else 0

    return {
        "length_url": len(full),
        "length_hostname": len(hostname),
        "ip": is_ip(hostname),
        "nb_dots": full.count("."),
        "nb_hyphens": full.count("-"),
        "nb_at": full.count("@"),
        "nb_qm": full.count("?"),
        "nb_and": full.count("&"),
        "nb_eq": full.count("="),
        "nb_underscore": full.count("_"),
        "nb_tilde": full.count("~"),
        "nb_percent": full.count("%"),
        "nb_slash": full.count("/"),
        "nb_star": full.count("*"),
        "nb_colon": full.count(":"),
        "nb_comma": full.count(","),
        "nb_semicolumn": full.count(";"),
        "nb_dollar": full.count("$"),
        "nb_space": full.count(" "),
        "nb_www": 1 if "www" in hostname else 0,
        "nb_com": full.count(".com"),
        "nb_dslash": full.count("//"),
        "http_in_path": 1 if "http" in path else 0,
        "https_token": 1 if "https" in full.replace("https://", "") else 0,
        "ratio_digits_url": digits_url / max(len(full), 1),
        "ratio_digits_host": digits_host / max(len(hostname), 1),
        "punycode": 1 if "xn--" in hostname else 0,
        "port": port_flag,
        "tld_in_path": 1 if suffix and (suffix in path) else 0,
        "tld_in_subdomain": 1 if suffix and (suffix in subdomain) else 0,
        "abnormal_subdomain": 1 if ("http" in subdomain or "https" in subdomain) else 0,
        "nb_subdomains": len(subdomain.split(".")) if subdomain else 0,
        "prefix_suffix": prefix_suffix,
        "random_domain": random_domain,
        "shortening_service": shortening_service,
        "path_extension": path_extension,
        "nb_redirection": nb_redirection,
        "nb_external_redirection": 0,
        "length_words_raw": len(re.findall(r"[A-Za-z0-9]+", full.lower())),
        "char_repeat": sum(1 for i in range(1, len(full)) if full[i] == full[i - 1]),
        "shortest_words_raw": 0,
        "shortest_word_host": 0,
        "shortest_word_path": 0,
        "longest_words_raw": 0,
        "longest_word_host": 0,
        "longest_word_path": 0,
        "avg_words_raw": 0.0,
        "avg_word_host": 0.0,
        "avg_word_path": 0.0,
        "phish_hints": sum(1 for k in PHISH_HINTS if k in full.lower()),
        "domain_in_brand": domain_in_brand,
        "brand_in_subdomain": brand_in_subdomain,
        "brand_in_path": brand_in_path,
        "suspicious_tld": suspicious_tld_flag,
        "statistical_report": statistical_report,
    }

def rule_based_eval(feats):
    very_important = {
        "suspicious_tld": (feats.get("suspicious_tld", 0) == 1),
        "nb_at": (feats.get("nb_at", 0) >= 1),
        "ip": (feats.get("ip", 0) == 1),
        "nb_underscore": (feats.get("nb_underscore", 0) > 3),
    }
    important = {
        "ratio_digits_url": (feats.get("ratio_digits_url", 0) > 0.3),
        "nb_subdomains": (feats.get("nb_subdomains", 0) > 3),
        "nb_percent": (feats.get("nb_percent", 0) > 5),
        "nb_tilde": (feats.get("nb_tilde", 0) >= 1),
        "nb_semicolumn": (feats.get("nb_semicolumn", 0) >= 1),
        "nb_star": (feats.get("nb_star", 0) >= 1),
        "nb_comma": (feats.get("nb_comma", 0) >= 1),
        "random_domain": (feats.get("random_domain", 0) == 1),
    }
    less_important = {
        "length_hostname": (feats.get("length_hostname", 0) > 30),
        "nb_dollar": (feats.get("nb_dollar", 0) >= 1),
        "nb_qm": (feats.get("nb_qm", 0) > 2),
        "nb_colon": (feats.get("nb_colon", 0) > 1),
        "nb_eq": (feats.get("nb_eq", 0) > 8),
        "nb_dots": (feats.get("nb_dots", 0) > 4),
        "nb_slash": (feats.get("nb_slash", 0) > 7),
        "nb_and": (feats.get("nb_and", 0) > 3),
        "nb_hyphens": (feats.get("nb_hyphens", 0) > 3),
        "http_in_path": (feats.get("http_in_path", 0) == 1),
        "https_token": (feats.get("https_token", 0) == 1),
        "port": (feats.get("port", 0) == 1),
        "shortening_service": (feats.get("shortening_service", 0) == 1),
    }

    vi_hits = [k for k, v in very_important.items() if v]
    imp_hits = [k for k, v in important.items() if v]
    less_hits = [k for k, v in less_important.items() if v]

    vi_count = len(vi_hits)
    imp_count = len(imp_hits)
    less_count = len(less_hits)

    risk_score = (3 * vi_count) + (2 * imp_count) + less_count
    rule_flag = int((vi_count >= 1) or (risk_score >= PREFILTER_HARD_PHISHING_SCORE))
    category = "Phishing" if rule_flag == 1 else "Suspicious"

    return risk_score, category, rule_flag, {
        "vi_count": vi_count,
        "imp_count": imp_count,
        "less_count": less_count,
        "vi_hits": vi_hits,
        "imp_hits": imp_hits,
        "less_hits": less_hits,
    }

def predict_url(url):
    feats = extract_url_features(url)
    return rule_based_eval(feats)

if data is not None:
    y_true = data["label"] if "label" in data.columns else None
    X_rules = data.drop(columns=["label"], errors="ignore")
    df_rules = X_rules.copy()

    def _col(c):
        return df_rules[c] if c in df_rules.columns else pd.Series(0, index=df_rules.index)

    very_important = {
        "suspicious_tld": (_col("suspicious_tld") == 1),
        "nb_at": (_col("nb_at") >= 1),
        "ip": (_col("ip") == 1),
        "nb_underscore": (_col("nb_underscore") > 3),
    }

    important = {
        "ratio_digits_url": (_col("ratio_digits_url") > 0.3),
        "nb_subdomains": (_col("nb_subdomains") > 3),
        "nb_percent": (_col("nb_percent") > 5),
        "nb_tilde": (_col("nb_tilde") >= 1),
        "nb_semicolumn": (_col("nb_semicolumn") >= 1),
        "nb_star": (_col("nb_star") >= 1),
        "nb_comma": (_col("nb_comma") >= 1),
        "random_domain": (_col("random_domain") == 1),
    }

    less_important = {
        "length_hostname": (_col("length_hostname") > 30),
        "nb_dollar": (_col("nb_dollar") >= 1),
        "nb_qm": (_col("nb_qm") > 2),
        "nb_colon": (_col("nb_colon") > 1),
        "nb_eq": (_col("nb_eq") > 8),
        "nb_dots": (_col("nb_dots") > 4),
        "nb_slash": (_col("nb_slash") > 7),
        "nb_and": (_col("nb_and") > 3),
        "nb_hyphens": (_col("nb_hyphens") > 3),
        "http_in_path": (_col("http_in_path") == 1),
        "https_token": (_col("https_token") == 1),
        "port": (_col("port") == 1),
        "shortening_service": (_col("shortening_service") == 1),
    }

    vi_count = pd.DataFrame(very_important).astype(int).sum(axis=1)
    imp_count = pd.DataFrame(important).astype(int).sum(axis=1)
    less_count = pd.DataFrame(less_important).astype(int).sum(axis=1)

    rule1_flag = (vi_count >= 1)
    risk_score = (3 * vi_count) + (2 * imp_count) + less_count
    rule_flag = ((rule1_flag) | (risk_score >= PREFILTER_HARD_PHISHING_SCORE)).astype(int)

    risk_category = np.where(rule_flag == 1, "Phishing", "Suspicious")

    distribusi = pd.Series(risk_category).value_counts().reset_index()
    distribusi.columns = ["Kategori Risiko", "Jumlah Data"]

    print("\n📊 DISTRIBUSI KATEGORI RISIKO:")
    display(distribusi)


if data is not None and y_true is not None:
    y_true_bin = pd.to_numeric(y_true, errors="coerce").fillna(0).astype(int)
    y_pred_rule = rule_flag.astype(int)

    cm = confusion_matrix(y_true_bin, y_pred_rule, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / cm.sum() if cm.sum() > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    cm_display = pd.DataFrame(
        cm,
        index=["Actual Benign (0)", "Actual Phishing (1)"],
        columns=["Predicted Benign (0)", "Predicted Phishing (1)"]
    )

    df_details = pd.DataFrame({
        "Komponen": ["True Negatives (TN)", "False Positives (FP)", "False Negatives (FN)", "True Positives (TP)"],
        "Jumlah Data": [tn, fp, fn, tp],
        "Indikator": ["✅ Tepat", "❌ Salah Alarm", "❌ Lolos Deteksi", "✅ Tepat"],
        "Penjelasan": [
            "URL aman ditebak aman",
            "URL aman ditebak phishing",
            "URL phishing ditebak aman",
            "URL phishing ditebak phishing"
        ]
    })

    df_metrics = pd.DataFrame({
        "Metrik Evaluasi": ["Akurasi (Accuracy)", "Presisi (Precision)", "Recall (Sensitivity)"],
        "Nilai": [f"{accuracy:.2%}", f"{precision:.2%}", f"{recall:.2%}"]
    })

    df_rule_cm = pd.DataFrame([{
        "vi_threshold": ">= 1",
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "total": int(cm.sum()),
        "flagged_phishing": int(y_pred_rule.sum()),
    }])

    print("\n🎯 HASIL EVALUASI RULE-BASED (vi_threshold >= 1)")
    print("-" * 55)

    print("\n📊 CONFUSION MATRIX:")
    display(cm_display)

    print("\n📈 RINCIAN PREDIKSI:")
    display(df_details)

    print("\n🚀 METRIK PERFORMA:")
    display(df_metrics)

    print("\n📝 TABEL RINGKASAN RAW:")
    display(df_rule_cm)


📊 DISTRIBUSI KATEGORI RISIKO:


,Kategori Risiko,Jumlah Data
0,Suspicious,9216
1,Phishing,2213



🎯 HASIL EVALUASI RULE-BASED (vi_threshold >= 1)
-------------------------------------------------------

📊 CONFUSION MATRIX:


,Predicted Benign (0),Predicted Phishing (1)
Actual Benign (0),5390,325
Actual Phishing (1),3826,1888



📈 RINCIAN PREDIKSI:


,Komponen,Jumlah Data,Indikator,Penjelasan
0,True Negatives (TN),5390,✅ Tepat,URL aman ditebak aman
1,False Positives (FP),325,❌ Salah Alarm,URL aman ditebak phishing
2,False Negatives (FN),3826,❌ Lolos Deteksi,URL phishing ditebak aman
3,True Positives (TP),1888,✅ Tepat,URL phishing ditebak phishing



🚀 METRIK PERFORMA:


,Metrik Evaluasi,Nilai
0,Akurasi (Accuracy),63.68%
1,Presisi (Precision),85.31%
2,Recall (Sensitivity),33.04%



📝 TABEL RINGKASAN RAW:


,vi_threshold,TN,FP,FN,TP,total,flagged_phishing
0,>= 1,5390,325,3826,1888,11429,2213


## **4. Train Model**

## **5. Confution Matrix**

In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# 1. FUNGSI EVALUASI KOMPREHENSIF
def evaluate_model(name, model, X_eval, y_eval):
    # Prediksi label dan probabilitas
    y_pred = model.predict(X_eval)
    
    # Ekstrak probabilitas (untuk ROC AUC) jika model mendukungnya
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    else:
        y_prob = y_pred

    # Hitung komponen Confusion Matrix
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
    
    # Hitung ROC AUC (dengan penanganan error jika probabilitas gagal diekstrak)
    try:
        auc = roc_auc_score(y_eval, y_prob)
    except Exception:
        auc = np.nan

    # Kembalikan dictionary hasil metrik
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1 Score": f1_score(y_eval, y_pred, zero_division=0),
        "AUC": auc if not np.isnan(auc) else np.nan,
        "✅ TP": int(tp),
        "✅ TN": int(tn),
        "❌ FP": int(fp),
        "❌ FN": int(fn)
    }

# 2. EKSEKUSI EVALUASI SEMUA MODEL
detail_rows = []

# Cek ketersediaan model sebelum dievaluasi (Urutan append menentukan urutan di tabel)
if "forest" in globals() or "forest" in locals():
    detail_rows.append(evaluate_model("Random Forest", forest, X_test, y_test))
    
if "xgb" in globals() or "xgb" in locals():
    detail_rows.append(evaluate_model("XGBoost", xgb, X_test, y_test))
    
if "stack_model" in globals() or "stack_model" in locals():
    detail_rows.append(evaluate_model("Stacking (RF+XGB+LR)", stack_model, X_test, y_test))

# 3. KOMPILASI & VISUALISASI HASIL (LEADERBOARD)
# Gabungkan hasil ke dalam DataFrame
detail_df = pd.DataFrame(detail_rows)

if not detail_df.empty:
    # Mengamankan urutan tetap: RF -> XGBoost -> Stacking
    detail_df = detail_df.reset_index(drop=True)

    # Styling DataFrame agar menjadi "Dashboard" yang interaktif
    styled_leaderboard = (
        detail_df.style
        # Format angka menjadi desimal (0.xxx) bukan persentase (%)
        .format({
            "Accuracy": "{:.3f}",
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1 Score": "{:.3f}",
            "AUC": "{:.3f}"
        })
        # Gradasi Hijau untuk skor metrik (semakin gelap = semakin baik)
        .background_gradient(cmap='Greens', subset=["Accuracy", "Precision", "Recall", "F1 Score", "AUC"])
        # Gradasi Biru untuk tebakan benar (TP & TN)
        .background_gradient(cmap='Blues', subset=["✅ TP", "✅ TN"])
        # Gradasi Merah untuk tebakan salah (FP & FN) -> Semakin gelap merahnya = semakin buruk
        .background_gradient(cmap='Reds', subset=["❌ FP", "❌ FN"])
        # Konfigurasi perataan teks
        .set_properties(**{'text-align': 'center', 'vertical-align': 'middle'})
    )

    # Menampilkan Output
    print("PERBANDINGAN PERFORMA KETIGA MODEL PADA DATA TEST ")
    display(styled_leaderboard)
else:
    print("⚠️ Tidak ada model yang terdeteksi untuk dievaluasi.")

⚠️ Tidak ada model yang terdeteksi untuk dievaluasi.


## **6. Implementasi Pipeline Rule-Based & Model ML (GA Tuning) untuk Evaluasi Matrix**

Pada bagian ini, pipeline prediksi dan evaluasi akan disusun ulang agar sesuai dengan logika di app.py, termasuk:


- Fungsi utility untuk feature extraction dan rule-based
- Pipeline prediksi model (Random Forest, XGBoost, Stacking/Meta) dengan parameter hasil tuning GA
- Evaluasi confusion matrix dan metrik utama (akurasi, presisi, recall, F1, AUC)


> **Catatan:**
> - Pastikan model sudah dilatih dengan parameter hasil tuning GA.
> - Jika ingin menguji URL baru, gunakan fungsi prediksi yang disediakan.


In [5]:
# --- Utility & Feature Extraction (app.py style) ---
import math
import ipaddress
import re
from urllib.parse import urlparse, urljoin
import tldextract
import numpy as np
import pandas as pd

# --- Konstanta dan List ---
SUSPICIOUS_TLD = [
    "zip", "xyz", "top", "tk", "ga", "ml", "gq", "cf", "pw", "cc", "club",
    "ws", "biz", "online", "site", "live", "work", "icu", "info",
    "cn", "ru", "loan", "download", "click"
]
STANDARD_PORTS = {21, 22, 23, 80, 443, 445, 1433, 1521, 3306, 3389}
SHORTENERS = {
    "bit.ly", "goo.gl", "tinyurl.com", "ow.ly", "t.co", "is.gd", "buff.ly",
    "adf.ly", "bit.do", "cutt.ly"
}
PHISH_HINTS = [
    "login", "verify", "update", "secure", "account", "bank",
    "paypal", "apple", "microsoft", "confirm", "signin", "password"
]
BRANDS = [
    "google", "facebook", "apple", "microsoft", "amazon", "paypal",
    "instagram", "whatsapp", "telegram", "netflix", "github", "linkedin"
]
PREFILTER_HARD_PHISHING_SCORE = 7

# --- Fungsi Utility ---
def entropy(s):
    if not s:
        return 0.0
    probs = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * math.log2(p) for p in probs)

def parse_url(url):
    u = (url or "").strip()
    if not u.startswith(("http://", "https://")):
        u = "http://" + u
    parsed = urlparse(u)
    hostname = (parsed.hostname or "").lower()
    path = parsed.path or ""
    return u, parsed, hostname, path

def is_ip(hostname):
    try:
        ipaddress.ip_address(hostname)
        return 1
    except Exception:
        return 0

def extract_url_features(url):
    full, parsed, hostname, path = parse_url(url)
    ext = tldextract.extract(full)
    subdomain = (ext.subdomain or "").lower()
    domain = (ext.domain or "").lower()
    suffix = (ext.suffix or "").lower()

    digits_url = sum(c.isdigit() for c in full)
    digits_host = sum(c.isdigit() for c in hostname)

    random_domain = 1 if entropy(domain) > 3.5 else 0
    shortening_service = 1 if any(s in hostname for s in SHORTENERS) else 0
    prefix_suffix = 1 if "-" in domain else 0
    path_extension = 1 if "." in path.split("/")[-1] else 0
    nb_redirection = max(full.lower().count("http") - 1, 0)

    try:
        parsed_port = parsed.port
    except ValueError:
        parsed_port = None
    port_flag = 1 if (parsed_port is not None and parsed_port not in STANDARD_PORTS) else 0

    tld_last_label = suffix.split(".")[-1] if suffix else ""
    suspicious_tld_flag = 1 if (suffix in SUSPICIOUS_TLD or tld_last_label in SUSPICIOUS_TLD) else 0

    domain_in_brand = 1 if any(b in domain for b in BRANDS) else 0
    brand_in_subdomain = 1 if any(b in subdomain for b in BRANDS) else 0
    brand_in_path = 1 if any(b in path.lower() for b in BRANDS) else 0

    statistical_report = 1 if (
        suspicious_tld_flag == 1
        or is_ip(hostname) == 1
        or full.count("@") >= 1
        or random_domain == 1
    ) else 0

    return {
        "length_url": len(full),
        "length_hostname": len(hostname),
        "ip": is_ip(hostname),
        "nb_dots": full.count("."),
        "nb_hyphens": full.count("-"),
        "nb_at": full.count("@"),
        "nb_qm": full.count("?"),
        "nb_and": full.count("&"),
        "nb_eq": full.count("="),
        "nb_underscore": full.count("_"),
        "nb_tilde": full.count("~"),
        "nb_percent": full.count("%"),
        "nb_slash": full.count("/"),
        "nb_star": full.count("*"),
        "nb_colon": full.count(":"),
        "nb_comma": full.count(","),
        "nb_semicolumn": full.count(";"),
        "nb_dollar": full.count("$"),
        "nb_space": full.count(" "),
        "nb_www": 1 if "www" in hostname else 0,
        "nb_com": full.count(".com"),
        "nb_dslash": full.count("//"),
        "http_in_path": 1 if "http" in path else 0,
        "https_token": 1 if "https" in full.replace("https://", "") else 0,
        "ratio_digits_url": digits_url / max(len(full), 1),
        "ratio_digits_host": digits_host / max(len(hostname), 1),
        "punycode": 1 if "xn--" in hostname else 0,
        "port": port_flag,
        "tld_in_path": 1 if suffix and (suffix in path) else 0,
        "tld_in_subdomain": 1 if suffix and (suffix in subdomain) else 0,
        "abnormal_subdomain": 1 if ("http" in subdomain or "https" in subdomain) else 0,
        "nb_subdomains": len(subdomain.split(".")) if subdomain else 0,
        "prefix_suffix": prefix_suffix,
        "random_domain": random_domain,
        "shortening_service": shortening_service,
        "path_extension": path_extension,
        "nb_redirection": nb_redirection,
        "nb_external_redirection": 0,
        "length_words_raw": len(re.findall(r"[A-Za-z0-9]+", full.lower())),
        "char_repeat": sum(1 for i in range(1, len(full)) if full[i] == full[i - 1]),
        "shortest_words_raw": 0,
        "shortest_word_host": 0,
        "shortest_word_path": 0,
        "longest_words_raw": 0,
        "longest_word_host": 0,
        "longest_word_path": 0,
        "avg_words_raw": 0.0,
        "avg_word_host": 0.0,
        "avg_word_path": 0.0,
        "phish_hints": sum(1 for k in PHISH_HINTS if k in full.lower()),
        "domain_in_brand": domain_in_brand,
        "brand_in_subdomain": brand_in_subdomain,
        "brand_in_path": brand_in_path,
        "suspicious_tld": suspicious_tld_flag,
        "statistical_report": statistical_report,
    }

# --- Rule-Based Evaluation ---
def rule_based_eval(feats):
    very_important = {
        "suspicious_tld": (feats.get("suspicious_tld", 0) == 1),
        "nb_at": (feats.get("nb_at", 0) >= 1),
        "ip": (feats.get("ip", 0) == 1),
        "nb_underscore": (feats.get("nb_underscore", 0) > 3),
    }
    important = {
        "ratio_digits_url": (feats.get("ratio_digits_url", 0) > 0.3),
        "nb_subdomains": (feats.get("nb_subdomains", 0) > 3),
        "nb_percent": (feats.get("nb_percent", 0) > 5),
        "nb_tilde": (feats.get("nb_tilde", 0) >= 1),
        "nb_semicolumn": (feats.get("nb_semicolumn", 0) >= 1),
        "nb_star": (feats.get("nb_star", 0) >= 1),
        "nb_comma": (feats.get("nb_comma", 0) >= 1),
        "random_domain": (feats.get("random_domain", 0) == 1),
    }
    less_important = {
        "length_hostname": (feats.get("length_hostname", 0) > 30),
        "nb_dollar": (feats.get("nb_dollar", 0) >= 1),
        "nb_qm": (feats.get("nb_qm", 0) > 2),
        "nb_colon": (feats.get("nb_colon", 0) > 1),
        "nb_eq": (feats.get("nb_eq", 0) > 8),
        "nb_dots": (feats.get("nb_dots", 0) > 4),
        "nb_slash": (feats.get("nb_slash", 0) > 7),
        "nb_and": (feats.get("nb_and", 0) > 3),
        "nb_hyphens": (feats.get("nb_hyphens", 0) > 3),
        "http_in_path": (feats.get("http_in_path", 0) == 1),
        "https_token": (feats.get("https_token", 0) == 1),
        "port": (feats.get("port", 0) == 1),
        "shortening_service": (feats.get("shortening_service", 0) == 1),
    }

    vi_hits = [k for k, v in very_important.items() if v]
    imp_hits = [k for k, v in important.items() if v]
    less_hits = [k for k, v in less_important.items() if v]

    vi_count = len(vi_hits)
    imp_count = len(imp_hits)
    less_count = len(less_hits)

    risk_score = (3 * vi_count) + (2 * imp_count) + less_count
    rule_flag = int((vi_count >= 1) or (risk_score >= PREFILTER_HARD_PHISHING_SCORE))
    category = "Phishing" if rule_flag == 1 else "Suspicious"

    return risk_score, category, rule_flag, {
        "vi_count": vi_count,
        "imp_count": imp_count,
        "less_count": less_count,
        "vi_hits": vi_hits,
        "imp_hits": imp_hits,
        "less_hits": less_hits,
    }


In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import time

# --- Parameter GA Tuning (Hasil Genetic Algorithm Optimization) ---
ga_rf_params = {
    "n_estimators": 428,
    "max_depth": 16,
    "min_samples_split": 4,
    "min_samples_leaf": 1,
    "max_features": "log2",
    "random_state": 12,
    "n_jobs": -1
}

ga_xgb_params = {
    "n_estimators": 356,
    "learning_rate": 0.24470280263728614,
    "max_depth": 5,
    "subsample": 0.799109701671054,
    "colsample_bytree": 0.7933724339946145,
    "min_child_weight": 6.656725584883891,
    "gamma": 1.295747350495099,
    "reg_alpha": 0.8880322724647041,
    "reg_lambda": 2.2701556904961406,
    "eval_metric": "logloss",
    "random_state": 12,
    "n_jobs": -1
}

# --- Training Models dengan GA Parameters ---
rf_ga = RandomForestClassifier(**ga_rf_params)
xgb_ga = XGBClassifier(**ga_xgb_params)

print("⏳ Training Random Forest (GA tuning)...")
t0 = time.time()
rf_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

print("⏳ Training XGBoost (GA tuning)...")
t0 = time.time()
xgb_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

# --- Stacking (Meta) dengan GA tuned models ---
base_learners_ga = [
    ("rf", rf_ga),
    ("xgb", xgb_ga),
]
meta_learner_ga = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=12)
stack_ga = StackingClassifier(
    estimators=base_learners_ga,
    final_estimator=meta_learner_ga,
    stack_method="predict_proba",
    n_jobs=-1
)
print("⏳ Training Stacking (GA tuning)...")
t0 = time.time()
stack_ga.fit(X_train, y_train)
print(f"Done in {time.time()-t0:.2f}s")

# --- Evaluasi pada data test ---
def evaluate_all_models(X_eval, y_eval):
    models = [
        ("Random Forest (GA)", rf_ga),
        ("XGBoost (GA)", xgb_ga),
        ("Stacking (RF+XGB+LR, GA)", stack_ga)
    ]
    results = []
    for name, model in models:
        y_pred = model.predict(X_eval)
        y_prob = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else y_pred
        tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()
        try:
            auc = roc_auc_score(y_eval, y_prob)
        except Exception:
            auc = np.nan
        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_eval, y_pred),
            "Precision": precision_score(y_eval, y_pred, zero_division=0),
            "Recall": recall_score(y_eval, y_pred, zero_division=0),
            "F1 Score": f1_score(y_eval, y_pred, zero_division=0),
            "AUC": auc,
            "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
        })
    return pd.DataFrame(results)

# --- Hasil Evaluasi ---
results_ga = evaluate_all_models(X_test, y_test)
print("\n📊 PERBANDINGAN MODEL DENGAN GA TUNING PARAMETERS:")
display(results_ga.style.format({"Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}", "F1 Score": "{:.3f}", "AUC": "{:.3f}"}))

⏳ Training Random Forest (GA tuning)...
Done in 0.83s
⏳ Training XGBoost (GA tuning)...
Done in 0.96s
⏳ Training Stacking (GA tuning)...
Done in 11.41s

📊 PERBANDINGAN MODEL DENGAN GA TUNING PARAMETERS:


,Model,Accuracy,Precision,Recall,F1 Score,AUC,TP,TN,FP,FN
0,Random Forest (GA),0.972,0.972,0.972,0.972,0.995,1111,1111,32,32
1,XGBoost (GA),0.974,0.974,0.974,0.974,0.996,1113,1113,30,30
2,"Stacking (RF+XGB+LR, GA)",0.973,0.973,0.974,0.973,0.996,1113,1112,31,30
